# Telco Customer Churn: EDA, Feature Engineering & Model Comparison
End-to-end supervised learning workflow comparing Logistic Regression, Random Forest, Gradient Boosting, XGBoost, and Multi-Layer Perceptron (ANN).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [2]:
# Load dataset and clean missing TotalCharges entries
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Charge_to_Tenure_Ratio'] = df['TotalCharges'] / (df['tenure'] + 1)

In [3]:
# Encode categorical variables and scale numerical features
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Charge_to_Tenure_Ratio']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [4]:
# Train and evaluate models using 5-fold Stratified Cross-Validation
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, eval_metric='logloss', random_state=42),
    'ANN (MLP)': MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=300, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    cv_score = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    print(f'{name} | Test ROC-AUC: {roc_auc_score(y_test, probs):.4f} | 5-Fold CV AUC: {cv_score:.4f}')